### LangGraph - 提示词助手
***
- 根据用户输入
- 生成提示词并优化

### 收集用户需求
***

In [1]:
from typing import List

from langchain_core.messages import SystemMessage
from langchain_deepseek import ChatDeepSeek
import os
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel

template = """
            你的工作是从用户那里获取有关他们想要创建的提示模板类型的信息。

            你应该从他们那里获取以下信息:

            - 提示词的目标是什么
            - 哪些变量将被传递到提示词模板中
            - 输出不应该做什么的任何约束
            - 输出必须遵守的任何要求

            如果你无法辨别这些信息，请要求他们澄清! 不要尝试胡乱猜测。

            在你能够辨别所有信息后，调用相关工具。
            """

def get_messages_info(messages):
    return [SystemMessage(content=template)] + messages

class PromptInstructions(BaseModel):
    """关于如何提示LLM的指令"""

    objective: str # 目标
    variables: List[str]  # 变量
    constraints: List[str]  # 约束
    requirements: List[str]  # 要求

llm = ChatDeepSeek(
    model = "deepseek-ai/DeepSeek-V4-Flash",
    temperature=0,
    api_key=os.getenv("ds_api_key"),
    base_url=os.getenv("ds_base_url")
)
llm_with_tool = llm.bind_tools([PromptInstructions])

def info_chain(state):
    messages = get_messages_info(state["messages"])
    response = llm_with_tool.invoke(messages)
    return {"messages": [response]}


生成提示词

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

# 新系统提示
prompt_system = """根据以下要求，编写一个好的提示词模板：
                
                {reqs}"""

# 获取提示消息的函数
# 智慧获取工具调用之后的消息
def get_prompt_messages(messages: List):
    tool_call = None
    other_msgs = []
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            tool_call = m.tool_calls[0]["args"]
        elif isinstance(m, ToolMessage):
            continue
        elif tool_call is not None:
            other_msgs.append(m)
    return [SystemMessage(content=prompt_system.format(reqs=tool_call))] + other_msgs

def prompt_gen_chain(state):
    messages = get_prompt_messages(state["messages"])
    response = llm.invoke(messages)
    return {"messages": [response]}




: 